
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png" alt="Databricks Learning">
</div>



# Monitor Data Quality

Each DLT Pipeline saves events and expectations metrics in the Storage Location defined on the pipeline. From this table we can see what is happening and the quality of the data passing through it.

You can leverage the expecations directly as a SQL table with Databricks SQL to track your expectation metrics and send alerts as required. 

This notebook extracts and analyses expectation metrics to build such KPIS.

You can find your metrics opening the Settings of your DLT pipeline, under `storage`

In [0]:
%run ./Includes/Classroom-Setup-05.5


## System table setup
We'll create a table based on the events log being saved by DLT. The system tables are stored under the storage location defined in your DLT pipeline settings.

In [0]:
display(dbutils.fs.ls(DA.paths.pipeline_event_logs))


## DLT expectation analysis
Delta live table tracks our data quality through expectations. These expectations are stored as technical tables without the DLT log events. We can create a view to simply analyze this information

In [0]:
import pyspark.sql.functions as F

( 
    spark.read.load(DA.paths.pipeline_event_logs)
        .sort(F.col("timestamp").desc())
        .createOrReplaceTempView("pipeline_event_logs")
)

In [0]:
%sql
SELECT * FROM pipeline_event_logs

## Event logs table structure
The `details` column contains metadata about each Event sent to the Event Log. There are different fields depending on what type of Event it is. Some examples include:
* `user_action` Events occur when taking actions like creating the pipeline
* `flow_definition` Events occur when a pipeline is deployed or updated and have lineage, schema, and execution plan information
  * `output_dataset` and `input_datasets` - output table/view and its upstream table(s)/view(s)
  * `flow_type` - whether this is a complete or append flow
  * `explain_text` - the Spark explain plan
* `flow_progress` Events occur when a data flow starts running or finishes processing a batch of data
  * `metrics` - currently contains `num_output_rows`
  * `data_quality` - contains an array of the results of the data quality rules for this particular dataset
    * `dropped_records`
    * `expectations`
      * `name`, `dataset`, `passed_records`, `failed_records`
      
We can leverage this information to track our table quality using SQL

In [0]:
%sql
SELECT
  id,
  timestamp,
  sequence,
  event_type,
  message,
  level,
  details
FROM
  pipeline_event_logs
ORDER BY
  timestamp ASC;

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW data_quality_metrics AS (
  SELECT 
    id,
    timestamp,
    details:flow_progress.metrics.num_output_rows AS output_records,
    details:flow_progress.data_quality.dropped_records,
    details:flow_progress.status AS status_update,
    explode(
      from_json(
        details:flow_progress.data_quality.expectations,
        'array<struct<dataset: string, failed_records: bigint, name: string, passed_records: bigint>>'
      )
    ) AS expectations
  FROM pipeline_event_logs
  where details:flow_progress.data_quality.expectations IS NOT null
  ORDER BY timestamp);
  
select * from data_quality_metrics


## Visualizing the Quality Metrics

Let's run a few queries to show the metrics we can display. Ideally, we should be using Databricks SQL to create SQL Dashboard and track all the data, but for this example we'll run a quick query in the dashboard directly:

In [0]:
%sql
SELECT 
    sum(expectations.failed_records) AS failed_records, 
    sum(expectations.passed_records) AS passed_records, 
    expectations.name 
FROM data_quality_metrics 
GROUP BY expectations.name

### Plotting failed record per expectations

* This visualization represents the data quality metrics with the number of passed and failed records for different expectations.


In [0]:
%sql
SELECT
  expectations.name,
  SUM(expectations.passed_records) AS passed_records,
  SUM(expectations.failed_records) AS failed_records
FROM data_quality_metrics
GROUP BY expectations.name
-- Optionally, you can order the results by a specific column
-- ORDER BY passed_records DESC;



### What's next?

We now have our data ready to be used for more advanced.

We can start creating our first DBSQL Dashboard monitoring our data quality & DLT pipeline health.


&copy; 2024 Databricks, Inc. All rights reserved.<br/>
Apache, Apache Spark, Spark and the Spark logo are trademarks of the 
<a href="https://www.apache.org/">Apache Software Foundation</a>.<br/>
<br/><a href="https://databricks.com/privacy-policy">Privacy Policy</a> | 
<a href="https://databricks.com/terms-of-use">Terms of Use</a> | 
<a href="https://help.databricks.com/">Support</a>